# HW2: Multiple Object Video Detection (VisDrone)

In [ ]:
import os, gc, json, time, csv, random, shutil, functools
from pathlib import Path

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import Rectangle
from PIL import Image, ImageDraw
from tqdm.auto import tqdm
from sklearn.cluster import KMeans
from sklearn.model_selection import GroupKFold

import torch
import torch.nn.functional as F
import torchvision
from torch.utils.data import DataLoader, Dataset
from torchvision.ops import batched_nms
from torchvision.models.detection import retinanet as retinanet_module, roi_heads as roi_heads_module
from torchvision.models.detection.anchor_utils import AnchorGenerator
from torchvision.models.detection.faster_rcnn import FastRCNNPredictor
from torchvision.models.detection.retinanet import RetinaNetClassificationHead
from torchvision.transforms.functional import pil_to_tensor
from faster_coco_eval import COCO, COCOeval_faster
from ultralytics import YOLO, RTDETR
from ultralytics.models.utils.loss import DETRLoss

In [ ]:
DATA = Path(os.environ.get('HW2_DATA', '.'))
OUT = Path('out')

PART = os.environ.get('HW2_PART', 'all')  # rcnn, yolo, detr, merge or all
SETS = OUT / f'sets_{PART}'
RUNS = OUT / 'runs'
FIGS = OUT / 'figs'
SUBS = OUT / 'submissions'
RES = OUT / f'results_{PART}.csv'
for d in (SETS, RUNS, FIGS, SUBS):
    d.mkdir(parents=True, exist_ok=True)

SEED = 42
EPOCHS = 12
TRAIN_STEP, VAL_STEP = 10, 5
CLASSES = ['person', 'bicycle', 'car', 'van', 'truck', 'tricycle', 'bus', 'motor']
NUM_CLASSES = len(CLASSES) + 1
MASK_COLOR = (123, 117, 104)

BIG_GPU = torch.cuda.get_device_properties(0).total_memory > 8e9
if 'HW2_BIG_GPU' in os.environ:  # force the batch sizes of the big (1) or small (0) setting, e.g. on a 12 GB card
    BIG_GPU = os.environ['HW2_BIG_GPU'] == '1'
RUN_CV = BIG_GPU
TV_BATCH = 4 if BIG_GPU else 2
LR = 0.00125 * TV_BATCH
WORKERS = 0 if os.name == 'nt' else min(8, max(2, os.cpu_count() // 2))
WORKERS = int(os.environ.get('HW2_WORKERS', WORKERS))  # dataloader workers for torchvision, 0 = main process

random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED); torch.cuda.manual_seed_all(SEED)
print('gpu', torch.cuda.get_device_name(0), 'big', BIG_GPU, 'data', DATA.resolve())

## 1. Data
Train and val are merged into one list (ids of val get an offset), it makes the sequence-level folds easier later.

In [ ]:
def load_split(split, offset):
    with open(DATA / split / 'annotations_coco.json') as f:
        c = json.load(f)
    for im in c['images']:
        im['id'] += offset
        im['split'] = split
        im['path'] = str(DATA / split / im['file_name'])
    for a in c['annotations']:
        a['id'] += offset
        a['image_id'] += offset
    return c

tr, va = load_split('train', 0), load_split('val', 10_000_000)
images = tr['images'] + va['images']
anns = tr['annotations'] + va['annotations']
categories = tr['categories']
TRAIN_CLIPS = sorted({im['video'] for im in tr['images']})
VAL_CLIPS = sorted({im['video'] for im in va['images']})
del tr, va

by_image = {}
for a in anns:
    by_image.setdefault(a['image_id'], []).append(a)

img_df = pd.DataFrame([{k: im[k] for k in ('id', 'video', 'split', 'width', 'height', 'frame_id')} for im in images])
ann_df = pd.DataFrame([{'image_id': a['image_id'], 'cat': a['category_id'], 'w': a['bbox'][2], 'h': a['bbox'][3],
                        'crowd': a.get('iscrowd', 0)} for a in anns]).merge(img_df, left_on='image_id', right_on='id')
ann_df['side'] = np.sqrt(ann_df.w * ann_df.h)
obj_df = ann_df[ann_df.crowd == 0]
print(len(images), 'frames,', len(obj_df), 'boxes,', (ann_df.crowd == 1).sum(), 'ignored regions')

Class balance, object sizes and objects per frame, then the main numbers per clip.

In [ ]:
fig, ax = plt.subplots(1, 3, figsize=(16, 4))
share = obj_df.groupby(['split', 'cat']).size().unstack(0)
(share / share.sum()).set_axis(CLASSES).plot.bar(ax=ax[0], title='class share')
ax[1].hist(obj_df.side, bins=np.logspace(0.5, 3, 50))
ax[1].set(xscale='log', title='object size, sqrt(area) px')
for v in (32, 96):
    ax[1].axvline(v, color='k', ls='--')
per_clip = obj_df.groupby(['video', 'image_id']).size().groupby('video').mean().sort_values()
per_clip.plot.barh(ax=ax[2], title='objects per frame by clip')
ax[2].set_yticks([])
plt.tight_layout(); plt.savefig(FIGS / 'eda.png', dpi=120); plt.show()

clip_table = obj_df.groupby(['split', 'video']).agg(width=('width', 'first'), boxes=('side', 'size'), median_side=('side', 'median'),
                                                    small=('side', lambda s: (s < 32).mean())).round(2)
clip_table['per_frame'] = per_clip.round(1).reindex(clip_table.index.get_level_values(1)).values
clip_table.groupby('split').describe().T.loc[(['per_frame', 'median_side', 'small'], ['min', '50%', 'max']), :]

Frame widths in train and val against test.

In [ ]:
test_df = pd.read_csv(DATA / 'test_images.csv')
print('test:', test_df.groupby('width').size().to_dict())
print('train+val:', img_df.groupby('width').size().to_dict())

Ground truth on a few val frames to check the annotations. Green boxes are objects, red areas are ignored regions.

In [ ]:
sample_clips = random.sample(VAL_CLIPS, 4)
fig, axes = plt.subplots(2, 2, figsize=(14, 8))
for ax, clip in zip(axes.flat, sample_clips):
    frames = [im for im in images if im['video'] == clip and (im['frame_id'] - 1) % VAL_STEP == 0]
    im = frames[len(frames) // 2]
    ax.imshow(Image.open(im['path']))
    for a in by_image.get(im['id'], []):
        x, y, w, h = a['bbox']
        c = 'red' if a.get('iscrowd') else 'lime'
        ax.add_patch(Rectangle((x, y), w, h, fill=bool(a.get('iscrowd')), alpha=.4 if a.get('iscrowd') else 1, color=c, lw=.7))
    ax.set_title(clip, fontsize=9); ax.axis('off')
plt.tight_layout(); plt.savefig(FIGS / 'gt_overlay.png', dpi=110); plt.show()

Object sizes after the torchvision resize (short side 800, long side at most 1333). The anchor sizes in section 7 come from these numbers.

In [ ]:
scale = np.minimum(800 / obj_df[['width', 'height']].min(axis=1), 1333 / obj_df[['width', 'height']].max(axis=1))
w, h = obj_df.w * scale, obj_df.h * scale
keep = (w >= 1) & (h >= 1)
side = np.sqrt(w[keep] * h[keep])
print('side percentiles 5/25/50/75/95:', np.percentile(side, [5, 25, 50, 75, 95]).round(1))
print('below 32 px: %.1f%%, below 16 px: %.1f%%' % (100 * (side < 32).mean(), 100 * (side < 16).mean()))
print('h/w 10/50/90:', np.percentile(h[keep] / w[keep], [10, 50, 90]).round(2))
xy = np.log(np.c_[w[keep], h[keep]])
xy = xy[np.random.choice(len(xy), min(100_000, len(xy)), replace=False)]
print('kmeans w,h:', np.exp(KMeans(5, n_init=3, random_state=SEED).fit(xy).cluster_centers_).round(1).tolist())

## 2. Dataset for training
Neighbouring frames are almost the same, so I keep every 10th frame of train and every 5th of val.
Ignored regions are painted over because the trainers have no "don't care" option.

The split is done by clip, so frames of one video are never in both train and val.

In [ ]:
def make_set(name, train_clips, val_clips):
    root = SETS / name
    if (root / 'data.yaml').exists():
        return root
    train_clips, val_clips = set(train_clips), set(val_clips)
    jobs = []
    for im in images:
        step = TRAIN_STEP if im['video'] in train_clips else VAL_STEP
        part = 'train' if im['video'] in train_clips else 'val' if im['video'] in val_clips else None
        if part and (im['frame_id'] - 1) % step == 0:
            jobs.append((part, im))
    for part in ('train', 'val'):
        (root / 'images' / part).mkdir(parents=True, exist_ok=True)
        (root / 'labels' / part).mkdir(parents=True, exist_ok=True)
    for part, im in tqdm(jobs, desc=name):
        stem = f"{im['video']}_{im['frame_id']:07d}"
        pic = Image.open(im['path']).convert('RGB')
        draw = ImageDraw.Draw(pic)
        W, H = im['width'], im['height']
        lines = []
        for a in by_image.get(im['id'], []):
            x, y, w, h = a['bbox']
            if a.get('iscrowd'):
                draw.rectangle([x, y, x + w, y + h], fill=MASK_COLOR)
            else:
                lines.append(f"{a['category_id'] - 1} {(x + w / 2) / W:.6f} {(y + h / 2) / H:.6f} {w / W:.6f} {h / H:.6f}")
        pic.save(root / 'images' / part / f'{stem}.jpg', quality=92)
        (root / 'labels' / part / f'{stem}.txt').write_text('\n'.join(lines))
    names = ', '.join(f'"{c}"' for c in CLASSES)
    (root / 'data.yaml').write_text(f'path: {root.resolve().as_posix()}\ntrain: images/train\nval: images/val\nnc: {len(CLASSES)}\nnames: [{names}]\n')
    return root

if PART != 'merge':
    make_set('main', TRAIN_CLIPS, VAL_CLIPS)

## 3. Evaluation and helpers

`val_gt` takes every 5th frame of the validation clips and `coco_eval` returns COCO mAP, AP50 and AP by object size. Ignored regions are `iscrowd=1`, so detections inside them do not count as errors. Every model is scored with these two functions.

`should_run` skips experiments that are already finished or belong to another `PART`. Parts let the experiments run on different GPUs: `rcnn`, `yolo`, `detr`, then `merge` builds the joint tables from all result files. The `gpu` column matters for fps, speed is only comparable within one machine. `keep_going` lets the next experiment start if one crashes, and `write_csv` also checks the file format.

In [ ]:
def val_gt(clips):
    clips = set(clips)
    imgs = [im for im in images if im['video'] in clips and (im['frame_id'] - 1) % VAL_STEP == 0]
    ids = {im['id'] for im in imgs}
    return {'images': imgs, 'categories': categories, 'annotations': [a for i in ids for a in by_image.get(i, [])]}


def coco_eval(preds, gt):
    coco_gt = COCO(gt)
    if not preds:
        return dict.fromkeys(['mAP', 'AP50', 'AP75', 'AP_small', 'AP_medium', 'AP_large'], 0.0)
    ev = COCOeval_faster(coco_gt, coco_gt.loadRes(preds), 'bbox')
    ev.params.maxDets = [1, 100, 500]
    ev.evaluate(); ev.accumulate(); ev.summarize()
    s = ev.stats
    return dict(mAP=s[0], AP50=s[1], AP75=s[2], AP_small=s[3], AP_medium=s[4], AP_large=s[5])


def collect(predict, gt):
    preds, t = [], time.time()
    for im in tqdm(gt['images'], desc='predict', leave=False):
        boxes, scores, labels = predict(im['path'])
        for (x1, y1, x2, y2), s, c in zip(boxes.tolist(), scores.tolist(), labels.tolist()):
            preds.append({'image_id': im['id'], 'category_id': int(c) + 1, 'bbox': [x1, y1, x2 - x1, y2 - y1], 'score': s})
    return preds, len(gt['images']) / (time.time() - t)


def save_result(name, family, m, fps, note=''):
    row = {'name': name, 'family': family, **{k: round(float(v), 4) for k, v in m.items()}, 'fps': round(fps, 1), 'gpu': torch.cuda.get_device_name(0), 'note': note}
    old = pd.read_csv(RES) if RES.exists() else pd.DataFrame(columns=row.keys())
    pd.concat([old[old.name != name], pd.DataFrame([row])], ignore_index=True).to_csv(RES, index=False)
    print(name, {k: row[k] for k in ('mAP', 'AP50', 'AP_small', 'fps')})


PARTS = {'rcnn': {'two-stage', 'one-stage anchor'}, 'yolo': {'anchor-free', 'pipeline', 'inference', 'cv'}, 'detr': {'query'}, 'merge': set()}

def should_run(name, family):
    if PART != 'all' and family not in PARTS[PART]:
        return False
    return not (RES.exists() and name in set(pd.read_csv(RES).name))


def load_results():
    frames = [pd.read_csv(f) for f in sorted(OUT.glob('results_*.csv'))]
    return pd.concat(frames, ignore_index=True).drop_duplicates('name', keep='last')


def cleanup():
    gc.collect()
    torch.cuda.empty_cache()


FAILED = []

def keep_going(fn):
    def wrapper(name, *args, **kwargs):
        try:
            return fn(name, *args, **kwargs)
        except Exception as e:
            FAILED.append(name)
            print('FAILED', name, repr(e)[:300])
            cleanup()
    return wrapper


def write_csv(name, predict):
    target = SUBS / f'{name}.csv'
    with open(target, 'w', newline='', encoding='utf-8') as f:
        w = csv.writer(f)
        w.writerow(['image_id', 'PredictionString'])
        for r in tqdm(test_df.itertuples(), total=len(test_df), desc=name, leave=False):
            b, s, l = predict(str(DATA / r.file_name))
            parts = [f'{c} {sc:.4f} {x1:.1f} {y1:.1f} {x2 - x1:.1f} {y2 - y1:.1f}' for (x1, y1, x2, y2), sc, c in zip(b.tolist(), s.tolist(), l.tolist()) if x2 - x1 >= .5 and y2 - y1 >= .5]
            w.writerow([r.image_id, ' '.join(parts)])
    check = pd.read_csv(target, keep_default_na=False)
    fields = check.PredictionString.str.split()
    assert len(check) == len(test_df) and check.image_id.is_unique
    assert fields.map(len).max() <= 3000 and (fields.map(len) % 6 == 0).all()
    assert all(int(p[i]) in range(len(CLASSES)) for p in fields for i in range(0, len(p), 6))
    assert all(float(p[i + 4]) > 0 and float(p[i + 5]) > 0 for p in fields for i in range(0, len(p), 6))
    return target

## 4. Torchvision training loop (Faster R-CNN, RetinaNet)

Training uses the painted frames with a horizontal flip, the models do the resize themselves. SGD with a 300 iteration warmup, the learning rate drops 10x at 2/3 and 11/12 of the epochs. After each epoch val mAP is computed and the best weights are kept.

`make_fastrcnn_loss` replaces the torchvision loss so that the Smooth L1 beta and label smoothing can be changed. In `build_retina` the number of anchors per location stays 9, so the pretrained box head still fits.

In [ ]:
class FrameSet(Dataset):
    def __init__(self, root):
        self.files = sorted((root / 'images' / 'train').glob('*.jpg'))
        self.labels = root / 'labels' / 'train'

    def __len__(self):
        return len(self.files)

    def __getitem__(self, i):
        path = self.files[i]
        img = pil_to_tensor(Image.open(path).convert('RGB')).float() / 255
        _, H, W = img.shape
        boxes, labels = [], []
        for line in (self.labels / f'{path.stem}.txt').read_text().splitlines():
            c, cx, cy, w, h = map(float, line.split())
            b = [(cx - w / 2) * W, (cy - h / 2) * H, (cx + w / 2) * W, (cy + h / 2) * H]
            if b[2] - b[0] >= 1 and b[3] - b[1] >= 1:
                boxes.append(b); labels.append(int(c) + 1)
        boxes = torch.tensor(boxes, dtype=torch.float32).reshape(-1, 4)
        if torch.rand(1) < .5:
            img = img.flip(-1)
            boxes[:, [0, 2]] = W - boxes[:, [2, 0]]
        return img, {'boxes': boxes, 'labels': torch.tensor(labels, dtype=torch.int64)}


def collate(batch):
    return tuple(zip(*batch))


def make_fastrcnn_loss(beta, smoothing):
    def loss(class_logits, box_regression, labels, targets):
        labels = torch.cat(labels)
        targets = torch.cat(targets)
        cls_loss = F.cross_entropy(class_logits, labels, label_smoothing=smoothing)
        pos = torch.where(labels > 0)[0]
        box_regression = box_regression.reshape(class_logits.shape[0], -1, 4)
        box_loss = F.smooth_l1_loss(box_regression[pos, labels[pos]], targets[pos], beta=beta, reduction='sum')
        return cls_loss, box_loss / labels.numel()
    return loss


def build_frcnn(rpn_train=2000, rpn_test=1000, beta=1 / 9, smoothing=0.0):
    m = torchvision.models.detection.fasterrcnn_resnet50_fpn_v2(
        weights='DEFAULT', min_size=800, max_size=1333, rpn_post_nms_top_n_train=rpn_train, rpn_post_nms_top_n_test=rpn_test,
        box_detections_per_img=500, box_score_thresh=0.01)
    m.roi_heads.box_predictor = FastRCNNPredictor(m.roi_heads.box_predictor.cls_score.in_features, NUM_CLASSES)
    roi_heads_module.fastrcnn_loss = make_fastrcnn_loss(beta, smoothing)
    return m


def build_retina(anchors=None, alpha=0.25, gamma=2.0):
    m = torchvision.models.detection.retinanet_resnet50_fpn_v2(
        weights='DEFAULT', min_size=800, max_size=1333, detections_per_img=500, score_thresh=0.01)
    if anchors:
        sizes = tuple(tuple(int(s * 2 ** (o / 3)) for o in range(3)) for s in anchors)
        m.anchor_generator = AnchorGenerator(sizes, ((0.5, 1.0, 2.0),) * len(sizes))
    n = m.anchor_generator.num_anchors_per_location()[0]
    norm = functools.partial(torch.nn.GroupNorm, 32)
    m.head.classification_head = RetinaNetClassificationHead(256, n, NUM_CLASSES, norm_layer=norm)
    retinanet_module.sigmoid_focal_loss = functools.partial(torchvision.ops.sigmoid_focal_loss, alpha=alpha, gamma=gamma)
    return m


@torch.no_grad()
def predict_tv(model, path):
    img = (pil_to_tensor(Image.open(path).convert('RGB')).float() / 255).cuda()
    with torch.autocast('cuda'):
        out = model([img])[0]
    keep = out['labels'] > 0
    return out['boxes'][keep].cpu().numpy(), out['scores'][keep].cpu().numpy(), out['labels'][keep].cpu().numpy() - 1


@keep_going
def run_tv(name, family, builder, note='', **cfg):
    if not should_run(name, family):
        return
    cleanup()
    run_dir = RUNS / name
    run_dir.mkdir(parents=True, exist_ok=True)
    gt = val_gt(VAL_CLIPS)
    data = FrameSet(SETS / 'main')
    loader = DataLoader(data, batch_size=TV_BATCH, shuffle=True, num_workers=WORKERS, collate_fn=collate, pin_memory=True)
    model = builder(**cfg).cuda()
    opt = torch.optim.SGD([p for p in model.parameters() if p.requires_grad], lr=LR, momentum=.9, weight_decay=1e-4)
    sched = torch.optim.lr_scheduler.MultiStepLR(opt, [int(EPOCHS * 2 / 3), int(EPOCHS * 11 / 12)], gamma=.1)
    scaler = torch.amp.GradScaler('cuda')
    best, log, start = -1, [], time.time()

    for epoch in range(1, EPOCHS + 1):
        model.train()
        total = 0
        for it, (imgs, targets) in enumerate(tqdm(loader, desc=f'{name} {epoch}', leave=False)):
            imgs = [i.cuda(non_blocking=True) for i in imgs]
            targets = [{k: v.cuda() for k, v in t.items()} for t in targets]
            warm = min(1, (it + 1) / 300) if epoch == 1 else 1
            for g in opt.param_groups:
                g['lr'] = sched.get_last_lr()[0] * warm
            with torch.autocast('cuda'):
                loss = sum(model(imgs, targets).values())
            opt.zero_grad(set_to_none=True)
            scaler.scale(loss).backward()
            scaler.step(opt); scaler.update()
            total += loss.item()
        sched.step()

        model.eval()
        preds, fps = collect(lambda p: predict_tv(model, p), gt)
        m = coco_eval(preds, gt)
        log.append({'epoch': epoch, 'loss': total / len(loader), 'minutes': (time.time() - start) / 60, **m})
        pd.DataFrame(log).to_csv(run_dir / 'metrics.csv', index=False)
        if m['mAP'] > best:
            best = m['mAP']
            torch.save(model.state_dict(), run_dir / 'best.pt')

    model.load_state_dict(torch.load(run_dir / 'best.pt', weights_only=True))
    model.eval()
    preds, fps = collect(lambda p: predict_tv(model, p), gt)
    metrics = coco_eval(preds, gt)
    json.dump(preds, open(run_dir / 'val_preds.json', 'w'))
    write_csv(name, lambda p: predict_tv(model, p))
    save_result(name, family, metrics, fps, note)

## 5. Ultralytics training (YOLO, RT-DETR)

Ultralytics handles augmentation and the training loop, mosaic is switched off for the last 2 epochs. After training the best weights go through the same `coco_eval` on the original val frames, not the painted ones.

In [ ]:
def predict_ul(model, img, imgsz, iou=.6):
    r = model.predict(img, imgsz=imgsz, conf=.01, iou=iou, max_det=500, verbose=False, half=True, device=0)[0].boxes
    return r.xyxy.cpu().numpy(), r.conf.cpu().numpy(), r.cls.cpu().numpy().astype(int)


def load_ul(name):
    return (RTDETR if name.startswith('rtdetr') else YOLO)(str(RUNS / name / 'weights' / 'best.pt'))


@keep_going
def run_ul(name, family, model_file, imgsz, batch, note='', data='main', val_clips=None, **extra):
    if not should_run(name, family):
        return
    cleanup()
    cls = RTDETR if 'rtdetr' in model_file else YOLO
    m = cls(model_file)
    m.train(data=str((SETS / data / 'data.yaml').resolve()), epochs=EPOCHS, imgsz=imgsz, batch=batch, workers=WORKERS or 2, seed=SEED,
            project=str(RUNS.resolve()), name=name, exist_ok=True, patience=0, plots=False, verbose=False, device=0, close_mosaic=2, **extra)
    best = load_ul(name)
    gt = val_gt(val_clips or VAL_CLIPS)
    preds, fps = collect(lambda p: predict_ul(best, p, imgsz), gt)
    metrics = coco_eval(preds, gt)
    json.dump(preds, open(RUNS / name / 'val_preds.json', 'w'))
    if family != 'cv':
        write_csv(name, lambda p: predict_ul(best, p, imgsz))
    save_result(name, family, metrics, fps, note)

## 6. Two-stage: Faster R-CNN

Runs: default settings, more RPN proposals (4000 in training, 2000 in testing), and a changed loss (Smooth L1 beta 0.05 with label smoothing 0.1).

In [ ]:
run_tv('frcnn_base', 'two-stage', build_frcnn, 'baseline')
run_tv('frcnn_rpn', 'two-stage', build_frcnn, 'rpn 4000/2000 proposals', rpn_train=4000, rpn_test=2000)
run_tv('frcnn_loss', 'two-stage', build_frcnn, 'smooth l1 beta .05, label smoothing .1', beta=.05, smoothing=.1)

`frame_errors` matches predictions with score at least 0.3 to ground truth at IoU 0.5 and counts misses and false positives per frame. A box centred in an ignored region is not a false positive. `show_failures` prints the counts by clip and class, then draws the worst frames from different clips (green is ground truth, red is prediction).

In [ ]:
def box_iou_np(a, b):
    lt = np.maximum(a[:, None, :2], b[None, :, :2])
    rb = np.minimum(a[:, None, 2:], b[None, :, 2:])
    wh = np.clip(rb - lt, 0, None)
    inter = wh[..., 0] * wh[..., 1]
    area = lambda x: (x[:, 2] - x[:, 0]) * (x[:, 3] - x[:, 1])
    return inter / (area(a)[:, None] + area(b)[None, :] - inter + 1e-9)


def frame_errors(preds, gt, conf=.3):
    per_image = {}
    for p in preds:
        if p['score'] >= conf:
            per_image.setdefault(p['image_id'], []).append(p)
    rows, missed = [], []
    for im in gt['images']:
        items = [a for a in by_image.get(im['id'], []) if not a.get('iscrowd')]
        crowd = [a['bbox'] for a in by_image.get(im['id'], []) if a.get('iscrowd')]
        ps = sorted(per_image.get(im['id'], []), key=lambda p: -p['score'])
        used, fp = set(), 0
        for p in ps:
            x, y, w, h = p['bbox']
            hit = -1
            if items:
                A = np.array([[x, y, x + w, y + h]])
                B = np.array([[a['bbox'][0], a['bbox'][1], a['bbox'][0] + a['bbox'][2], a['bbox'][1] + a['bbox'][3]] for a in items])
                ious = box_iou_np(A, B)[0]
                for j in np.argsort(-ious):
                    if ious[j] < .5:
                        break
                    if j not in used and items[j]['category_id'] == p['category_id']:
                        hit = j
                        break
            if hit >= 0:
                used.add(hit)
            elif not any(cx <= x + w / 2 <= cx + cw and cy <= y + h / 2 <= cy + ch for cx, cy, cw, ch in crowd):
                fp += 1
        for j, a in enumerate(items):
            if j not in used:
                missed.append({'video': im['video'], 'cat': CLASSES[a['category_id'] - 1], 'side': np.sqrt(a['area'])})
        rows.append({'id': im['id'], 'video': im['video'], 'path': im['path'], 'gt': len(items), 'fn': len(items) - len(used), 'fp': fp})
    return pd.DataFrame(rows), pd.DataFrame(missed)


def show_failures(name, n=3):
    gt = val_gt(VAL_CLIPS)
    preds = json.load(open(RUNS / name / 'val_preds.json'))
    frames, missed = frame_errors(preds, gt)
    print(frames.groupby('video')[['gt', 'fn', 'fp']].sum())
    if len(missed):
        print(missed.groupby('cat').side.describe()[['count', '50%']].round(1))
    worst = frames.assign(err=frames.fn + frames.fp).sort_values('err', ascending=False).drop_duplicates('video').head(n)
    fig, axes = plt.subplots(1, n, figsize=(7 * n, 5), squeeze=False)
    for ax, row in zip(axes[0], worst.itertuples()):
        ax.imshow(Image.open(row.path))
        for a in by_image.get(row.id, []):
            if not a.get('iscrowd'):
                ax.add_patch(Rectangle(a['bbox'][:2], *a['bbox'][2:], fill=False, color='lime', lw=.7))
        for p in preds:
            if p['image_id'] == row.id and p['score'] >= .3:
                ax.add_patch(Rectangle(p['bbox'][:2], *p['bbox'][2:], fill=False, color='red', lw=.7))
        ax.set_title(f'{row.video} fn={row.fn} fp={row.fp}', fontsize=9); ax.axis('off')
    plt.tight_layout(); plt.savefig(FIGS / f'failures_{name}.png', dpi=110); plt.show()

if (RUNS / 'frcnn_base' / 'val_preds.json').exists():
    show_failures('frcnn_base')

## 7. One-stage anchor-based: RetinaNet
Anchors start from 16 px, about half of the boxes are smaller than the default 32 px anchor.

Runs: default settings, anchors from 16 to 256 px, and focal loss with gamma 1.5 and alpha 0.35.

In [ ]:
run_tv('retina_base', 'one-stage anchor', build_retina, 'baseline')
run_tv('retina_anchors', 'one-stage anchor', build_retina, 'anchors from 16 px', anchors=[16, 32, 64, 128, 256])
run_tv('retina_focal', 'one-stage anchor', build_retina, 'focal gamma 1.5 alpha .35', gamma=1.5, alpha=.35)

## 8. Anchor-free: YOLO11

Runs: baseline at 960, input size 1280, and higher box, cls and dfl loss weights. The batch size depends on GPU memory.

In [ ]:
y960 = 16 if BIG_GPU else 4
y1280 = 8 if BIG_GPU else 2
run_ul('yolo_base', 'anchor-free', 'yolo11s.pt', 960, y960, 'baseline')
run_ul('yolo_1280', 'anchor-free', 'yolo11s.pt', 1280, y1280, 'imgsz 1280')
run_ul('yolo_box', 'anchor-free', 'yolo11s.pt', 960, y960, 'box 10, cls .7, dfl 2', box=10.0, cls=0.7, dfl=2.0)

## 9. Query-based: RT-DETR

Runs: baseline at 640, AdamW with lr 2e-4, and changed loss weights (bbox 2, giou 5).

In [ ]:
d640 = 16 if BIG_GPU else 4
run_ul('rtdetr_base', 'query', 'rtdetr-l.pt', 640, d640, 'baseline')
run_ul('rtdetr_lr', 'query', 'rtdetr-l.pt', 640, d640, 'adamw lr 2e-4', optimizer='AdamW', lr0=2e-4, lrf=.1)

detr_init = DETRLoss.__init__

def detr_loss_weights(**gain):
    def init(self, *args, **kwargs):
        detr_init(self, *args, **kwargs)
        self.loss_gain.update(gain)
    DETRLoss.__init__ = init

detr_loss_weights(bbox=2, giou=5)
run_ul('rtdetr_loss', 'query', 'rtdetr-l.pt', 640, d640, 'loss weights bbox 2, giou 5')
DETRLoss.__init__ = detr_init

Validation mAP per epoch for YOLO and RT-DETR, taken from the Ultralytics training logs.

In [ ]:
plt.figure(figsize=(7, 4))
for name in ('yolo_base', 'rtdetr_base'):
    f = RUNS / name / 'results.csv'
    if f.exists():
        d = pd.read_csv(f)
        plt.plot(d['epoch'], d['metrics/mAP50-95(B)'], marker='o', label=name)
plt.xlabel('epoch'); plt.ylabel('val mAP'); plt.legend(); plt.savefig(FIGS / 'query_vs_cnn.png', dpi=120); plt.show()

## 10. Comparison of all models

A table of all runs, then AP by object size for the best run of each family.

In [ ]:
results = load_results()
display(results.sort_values(['family', 'mAP'], ascending=[True, False]))

best_per_family = results.loc[results[results.family.isin(['two-stage', 'one-stage anchor', 'anchor-free', 'query'])].groupby('family').mAP.idxmax()]
best_per_family.set_index('name')[['AP_small', 'AP_medium', 'AP_large']].plot.bar(figsize=(8, 4), title='AP by object size')
plt.tight_layout(); plt.savefig(FIGS / 'by_size.png', dpi=120); plt.show()

## 11. Final pipeline
A bigger model first, then inference changes on top of the best YOLO: tiles and soft-NMS.

In [ ]:
run_ul('final_yolo', 'pipeline', 'yolo11m.pt' if BIG_GPU else 'yolo11s.pt', 1280 if BIG_GPU else 960, 6 if BIG_GPU else 2,
       'bigger model')

Tiling runs the model on the full image and on 4 overlapping crops, each 60% of the width and height. Boxes are moved back to full image coordinates and merged with NMS or soft-NMS, then the top 500 are kept.

In [ ]:
def soft_nms(boxes, scores, sigma=.5, min_score=.01):
    scores = scores.copy()
    idx = np.arange(len(scores))
    keep = []
    while len(idx):
        i = idx[scores[idx].argmax()]
        keep.append(i)
        idx = idx[idx != i]
        if not len(idx):
            break
        ious = box_iou_np(boxes[[i]], boxes[idx])[0]
        scores[idx] *= np.exp(-ious ** 2 / sigma)
        idx = idx[scores[idx] > min_score]
    return np.array(keep), scores


def predict_tiled(model, path, imgsz, soft=False):
    pic = Image.open(path).convert('RGB')
    W, H = pic.size
    tw, th = int(W * .6), int(H * .6)
    parts = [predict_ul(model, pic, imgsz)]
    for x0 in (0, W - tw):
        for y0 in (0, H - th):
            b, s, l = predict_ul(model, pic.crop((x0, y0, x0 + tw, y0 + th)), imgsz)
            parts.append((b + np.array([x0, y0, x0, y0], dtype=np.float32), s, l))
    boxes = np.concatenate([p[0] for p in parts])
    scores = np.concatenate([p[1] for p in parts])
    labels = np.concatenate([p[2] for p in parts])
    if soft:
        out = []
        for c in np.unique(labels):
            ids = np.where(labels == c)[0]
            ids = ids[np.argsort(-scores[ids])[:2000]]
            k, s = soft_nms(boxes[ids], scores[ids])
            out.append((boxes[ids][k], s[k], labels[ids][k]))
        boxes, scores, labels = [np.concatenate(v) for v in zip(*out)] if out else (boxes, scores, labels)
    else:
        k = batched_nms(torch.tensor(boxes), torch.tensor(scores), torch.tensor(labels), .6).numpy()
        boxes, scores, labels = boxes[k], scores[k], labels[k]
    top = np.argsort(-scores)[:500]
    return boxes[top], scores[top], labels[top]

The three inference variants on val for the best YOLO: full image, tiles with NMS, tiles with soft-NMS.

In [ ]:
if PART in ('all', 'yolo'):
    results = load_results()
    yolos = results[results.family.isin(['anchor-free', 'pipeline'])].sort_values('mAP', ascending=False)
    best_name = yolos.iloc[0]['name']
    best_imgsz = 1280 if best_name == 'yolo_1280' or (best_name == 'final_yolo' and BIG_GPU) else 960
    model = load_ul(best_name)
    gt = val_gt(VAL_CLIPS)
    strategies = {
        'full': lambda p: predict_ul(model, p, best_imgsz),
        'tiles_nms': lambda p: predict_tiled(model, p, best_imgsz),
        'tiles_soft': lambda p: predict_tiled(model, p, best_imgsz, soft=True),
    }
    for key, fn in strategies.items():
        if should_run(f'{best_name}+{key}', 'inference'):
            preds, fps = collect(fn, gt)
            # kept so that ensemble.py can check the exact variant that goes to the leaderboard
            json.dump(preds, open(RUNS / best_name / f'val_preds_{key}.json', 'w'))
            save_result(f'{best_name}+{key}', 'inference', coco_eval(preds, gt), fps, key)
    results = load_results()
    display(results[results.family == 'inference'])

## 12. Sequence-level cross-validation of the YOLO baseline

3 folds by clip over all train and val clips. The dataset of each fold is deleted after its run. Only runs when `RUN_CV` is on.

In [ ]:
if RUN_CV and PART in ('all', 'yolo'):
    clips = np.array(TRAIN_CLIPS + VAL_CLIPS)
    for i, (a, b) in enumerate(GroupKFold(3).split(clips, groups=clips)):
        make_set(f'fold{i}', clips[a], clips[b])
        run_ul(f'cv_yolo_{i}', 'cv', 'yolo11s.pt', 960, y960, f'fold {i}', data=f'fold{i}', val_clips=clips[b].tolist())
        shutil.rmtree(SETS / f'fold{i}')
    cv = pd.read_csv(RES).query("family == 'cv'")
    print('cv mAP %.4f +- %.4f' % (cv.mAP.mean(), cv.mAP.std()))

## 13. CSV for the leaderboard

A CSV is already saved for every model right after its training. Here the tiled variants of the best YOLO are added, and the table shows which variant was best on val.

In [ ]:
if PART in ('all', 'yolo'):
    ranked = pd.read_csv(RES).query("family == 'inference'").sort_values('mAP', ascending=False)
    display(ranked[['name', 'mAP', 'AP50', 'AP_small']])
    for key, fn in strategies.items():
        if key != 'full':  # the full image csv was written right after training
            write_csv(f'{best_name}_{key}', fn)

shutil.rmtree(SETS, ignore_errors=True)
print(len(list(SUBS.glob('*.csv'))), 'csv files in', SUBS)
print('failed runs:', FAILED)